<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex09.2-channel-flow/Ex09.2_pinn_flow_past_tube.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_09.2 — The Mean Flow Past a Tube

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L9.2 · Turbulent Flow**

Water runs through a duct and past a tube that crosses it. How much pressure
does the tube cost, and what force does the water put on it? The flow is
turbulent, so what can be solved for is its mean, and that needs a model of
the turbulence: an eddy viscosity. You solve the mean flow three ways - a
finite-element reference, finite volumes on a grid, and a physics-informed
network with the stream function - and compare them on the two numbers an
engineer wants and on time. Then the same network finds the eddy viscosity
from twelve velocity probes.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and finite volumes on a grid for an agreed 5 % | — |
| **5** | a PINN with the stream function, and the comparison | the trial stream function, the residuals |
| **6** | the inverse problem: the eddy viscosity from twelve probes | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About six minutes on a CPU. Numbers are printed with at most two decimals.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex09.2-channel-flow/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex09.2_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

Water runs through a duct 100 mm high at a mean speed of 0.25 m/s. A tube of
40 mm crosses the duct, on its centreline, 120 mm from the inlet. Two numbers
are wanted: the **pressure drop** over the 400 mm of duct, which the pump has
to supply, and the **drag** on the tube, which its supports have to hold.

## 2 · The data

| | |
|---|---|
| duct | 100 mm high, 400 mm long; walls with no slip |
| water, 20 °C | $\rho$ = 998 kg/m³, $\nu$ = 1.0e-06 m²/s |
| inflow | mean speed $U$ = 0.25 m/s |
| tube | 40 mm across, centre 120 mm from the inlet, on the centreline |
| Reynolds number | $UH/\nu$ = 25 000: **turbulent** |
| closure | one eddy viscosity for the whole duct, $\nu + \nu_t$ = 5.0e-04 m²/s |

## 3 · The physics

At a Reynolds number of 25 000 the flow is turbulent, and no solver here can
resolve its eddies. What can be solved for is the **mean flow**. Averaging the
flow equations leaves one term that no equation determines, the Reynolds
stresses (L9.2's *Closure Problem*). The **eddy-viscosity** closure models
them as extra viscosity, and the mean flow then obeys the laminar equations
with $\nu$ replaced by $\nu_{\mathrm{eff}} = \nu + \nu_t$:

$$(\mathbf{u}\cdot\nabla)\,\mathbf{u} = -\frac{1}{\rho}\nabla p + \nu_{\mathrm{eff}}\,\nabla^2\mathbf{u},
  \qquad \nabla\cdot\mathbf{u} = 0$$

**The closure of this notebook is one number**: $\nu_t = 0.02\,UH$ everywhere,
five hundred times the viscosity of water. It is a choice, made so that the
modelled mean flow is steady and smooth. A real eddy viscosity varies from
place to place, and the answer below is only as good as that number - which
is why section 6 finds it from measurements instead.

**Scaled units.** Lengths over the duct height $H$, speeds over $U$, pressure
over $\rho U^2$ = 62.38 Pa. The duct is 4 × 1, the tube a circle of radius 0.2
at (1.2, 0.5), the inflow $u = 6y(1-y)$, and one number is left:
$\nu_{\mathrm{eff}}/(UH)$ = 0.02. A force per metre of tube is in units of
$\rho U^2 H$ = 6.24 N/m. Everything below is solved in these units, and the
two answers are printed in pascal and in newton per metre. The cell below
prints the data and draws the duct, with the network's points and the twelve
probes of section 6.

In [ ]:
pb.describe_problem()
pts_np = pb.sample_fluid(2000)                                # the network's collocation points, drawn once
t = np.linspace(0, 2 * np.pi, 200)
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(pts_np[:, 0], pts_np[:, 1], ".", ms=2, color="0.6", label="collocation points")
ax.plot([0, 4], [0, 0], "k", lw=2); ax.plot([0, 4], [1, 1], "k", lw=2, label="walls, no slip")
ax.plot(pb.TUBE["xc"] + pb.TUBE["r"] * np.cos(t), pb.TUBE["yc"] + pb.TUBE["r"] * np.sin(t), color="tab:orange", lw=2, label="tube, no slip")
ax.plot(pb.PROBES[:, 0], pb.PROBES[:, 1], "k^", ms=7, label="velocity probes (section 6)")
ax.set_aspect("equal"); ax.set_xlim(0, 4); ax.set_xlabel("x / H   (inflow on the left)"); ax.set_ylabel("y / H")
ax.legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=4)
plt.tight_layout(); plt.show()

**What you should see.** The data of section 2: a Reynolds number of 25 000,
an effective viscosity of 5.0e-04 m²/s, 500 times water's, a pressure scale of
62.38 Pa and a force scale of 6.24 N per metre of tube. The model's own
Reynolds number is 50. In the drawing half of the points are spread over the
duct and half round the tube and its wake.

---

## 4 · The reference, and finite volumes

**The reference.** The problem has no exact solution, so the methods are
scored against a **converged numerical solution**: finite elements with a
quadratic velocity and a linear pressure, on a mesh **fitted to the tube** and
refined round it, the nonlinear system solved by Newton's method.
`pb.reference(spacing)` builds it; read it in `problem.py` if you want to see
how. The cell below halves the node spacing twice and prints the two answers
each time.

In [ ]:
g = np.linspace(0.02, 3.98, 199); gy = np.linspace(0.01, 0.99, 99); GX, GY = np.meshgrid(g, gy)   # points to compare fields on
fields = {}
for n in (10, 20, 40):
    t0 = time.perf_counter(); r = pb.reference(1 / n); sec = time.perf_counter() - t0
    fields[n] = (r, pb.reference_at(r, GX, GY))
    print(f"  spacing H/{n}: {r['n_unknowns']:6d} unknowns, pressure drop {pb.P_SCALE * r['dp']:.2f} Pa, "
          f"drag {pb.F_SCALE * r['drag']:.2f} N/m (coefficient {pb.drag_coefficient(r['drag']):.2f}), {sec:.2f} s")
for a, b in ((10, 20), (20, 40)):
    du = np.nanmax(np.hypot(fields[a][1][0] - fields[b][1][0], fields[a][1][1] - fields[b][1][1]))
    print(f"  H/{a} against H/{b}: pressure drop {100 * abs(fields[a][0]['dp'] / fields[b][0]['dp'] - 1):.3f} %, "
          f"drag {100 * abs(fields[a][0]['drag'] / fields[b][0]['drag'] - 1):.3f} %, velocity at most {du:.2e} U apart")
ref, (u_ref, v_ref, p_ref) = fields[40]                       # the reference from here on
water = ~np.isnan(u_ref)                                      # True in the water, False inside the tube

**What you should see.** A pressure drop of 218.25 Pa and a drag of 11.40
N per metre of tube (a drag coefficient of 9.14 on the mean speed), the
same to four figures on the last two meshes: they differ by 0.006 % and
0.004 %. The velocity fields differ by at most 7.68e-02 and then 1.59e-02
of the mean speed. Measured once on a mesh twice as fine again, H/40 is within
0.004 $U$ everywhere. **That is far better than the accuracy agreed below**,
so H/40 is the reference. It takes about five seconds.

**Finite volumes.** The method of most flow codes: a grid of square cells,
the velocity on the cell faces and the pressure at the centres, marched in
time until the flow stops changing. `pb.finite_volumes(n_y)` is that solver,
with central differences. The grid does not fit the tube: the tube is a
**staircase**, every face inside it held at zero - as in Ex_08.1.

**The agreed accuracy is 5 %** on the pressure drop and on the drag: the two
numbers that were asked for. The cell below finds the coarsest grid that
meets it.

In [ ]:
TARGET = 5.0                                                  # per cent: the agreed accuracy
for n_y in (20, 40, 80):
    t0 = time.perf_counter(); fv = pb.finite_volumes(n_y); t_fv = time.perf_counter() - t0
    e_dp, e_drag = 100 * (fv["dp"] / ref["dp"] - 1), 100 * (fv["drag"] / ref["drag"] - 1)
    print(f"  {4 * n_y:3d} x {n_y} cells: pressure drop {pb.P_SCALE * fv['dp']:.2f} Pa ({e_dp:+.2f} %), "
          f"drag {pb.F_SCALE * fv['drag']:.2f} N/m ({e_drag:+.2f} %), steady after {fv['t']:.1f} H/U, {t_fv:.2f} s"
          + ("   <- meets 5 %" if max(abs(e_dp), abs(e_drag)) <= TARGET else ""))
N_Y = 80                                                      # the coarsest grid that meets it; fv is that solution

**What you should see.** Both numbers too low, and the error halving with the
spacing - first order, as on the staircase of Ex_08.1: the pressure drop
10.26, 5.61 and 2.86 % off and the drag 13.64, 7.76 and 4.03 %
off. The staircase is the reason: every face inside the tube is counted as
solid and every face outside as water, so the tube the grid sees is up to one
cell too small. It takes **320 × 80 cells**, and about a quarter of a minute, to come within 5 %.

---

## 5 · A physics-informed network, and the comparison

**The network** takes $(x, y)$ and returns two numbers, a stream function part
and the pressure. It is four layers of 32 tanh neurons, trained on the 2000
collocation points drawn above.

**Incompressibility is built in.** The velocity is taken from a **stream
function** $\psi$ as $u = \psi_y$, $v = -\psi_x$, so $\nabla\cdot\mathbf{u} =
0$ holds exactly and there is no continuity term in the loss (L9.1's *Stream
Function*).

**The walls and the inflow are built in too.** The trial stream function is

$$\hat\psi(x, y) = \underbrace{3y^2 - 2y^3}_{\text{the inflow}} \;+\; \tanh^2(2x)\;\big(y(1-y)\big)^2\;\mathcal{N}_\psi(x, y)$$

The first term is the stream function of the inflow: its $y$-derivative is
$6y(1-y)$. The factor $(y(1-y))^2$ and its derivative vanish on both walls, so
$u = v = 0$ there whatever the network does; $\tanh^2(2x)$ and its derivative
vanish at the inlet, so the inflow is exact.

**The tube and the outlet stay loss terms**: no slip on 200 points of the
tube's surface, with the weight $w_T$ = 10, and $p = 0$, $v = 0$ on the outlet.

**TODO 1** is the trial stream function; **TODO 2** the two momentum
residuals, in scaled units:
$u\,u_x + v\,u_y + p_x - \nu_{\mathrm{eff}}(u_{xx} + u_{yy})$ and the same for
$v$ with $p_y$.

In [ ]:
class Flow(torch.nn.Module):                                  # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=2, n_out=2, n_hidden=32, n_layers=4)   # dense tanh network: x, y in; a stream part and p out
    def forward(self, xy):
        x, y = xy[:, 0:1], xy[:, 1:2]
        out = self.net(xy)                                    # column 0: the stream part, column 1: the pressure
        # TODO 1: the trial stream function - the inflow's stream function, plus tanh(2x) squared
        # times (y (1 - y)) squared times the network's first output.
        #
        #   psi = pb.psi_inflow(y) + torch.tanh(2 * x) ** 2 * (y * (1 - y)) ** 2 * out[:, 0:1]
        raise NotImplementedError("Write the trial stream function")
        return psi, out[:, 1:2]
    def fields(self, xy):
        psi, p = self(xy)
        g = grad(psi, xy)                                     # the derivatives of psi by autograd
        return g[:, 1:2], -g[:, 0:1], p                       # u = psi_y, v = -psi_x: divergence-free by construction

In [ ]:
# TODO 2: the two momentum residuals of the mean flow, in scaled units.
#
#   def residuals(flow, xy, nu):
#       u, v, p = flow.fields(xy)
#       gu, gv, gp = grad(u, xy), grad(v, xy), grad(p, xy)        # columns x, y
#       lap_u = grad(gu[:, 0:1], xy)[:, 0:1] + grad(gu[:, 1:2], xy)[:, 1:2]
#       lap_v = grad(gv[:, 0:1], xy)[:, 0:1] + grad(gv[:, 1:2], xy)[:, 1:2]
#       res_u = u * gu[:, 0:1] + v * gu[:, 1:2] + gp[:, 0:1] - nu * lap_u
#       res_v = u * gv[:, 0:1] + v * gv[:, 1:2] + gp[:, 1:2] - nu * lap_v
#       return res_u, res_v
raise NotImplementedError("Write residuals(flow, xy, nu)")

In [ ]:
W_T = 10.0                                                    # the weight of no slip on the tube
pts = to_tensor(pts_np, requires_grad=True)                   # the collocation points, differentiable
tube = to_tensor(pb.sample_tube(200), requires_grad=True)     # 200 points on the tube's surface
outlet = to_tensor(pb.sample_outlet(101), requires_grad=True) # 101 points on the outlet

def boundary_loss(flow):
    ut, vt, _ = flow.fields(tube)                             # the velocity on the tube: should be zero
    _, vo, po = flow.fields(outlet)                           # on the outlet: no cross-flow, and the pressure's level
    return W_T * (ut.pow(2).mean() + vt.pow(2).mean()) + vo.pow(2).mean() + po.pow(2).mean()

flow = Flow().to(DEVICE)                                      # four layers of 32
describe(flow.net, n_collocation=len(pts))                    # the parameter count and the collocation-to-parameter ratio
def loss_fn():
    res_u, res_v = residuals(flow, pts, pb.NU_EFF)            # the two momentum residuals at the collocation points
    return res_u.pow(2).mean() + res_v.pow(2).mean() + boundary_loss(flow)   # the composite loss
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(flow, loss_fn, adam_steps=300, lbfgs_steps=60, lr=2e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
def on_grid(net):
    # a network's u, v, p at the comparison points that lie in the water
    q = to_tensor(np.c_[GX[water], GY[water]], requires_grad=True)   # differentiable: the velocity is a derivative
    return [to_numpy(f).ravel() for f in net.fields(q)]
u_pin, v_pin, p_pin = on_grid(flow)
t_pin = middle(lambda: on_grid(flow), reps=3)
drag_pin, dp_pin = pb.network_numbers(flow.fields)            # the drag and the pressure drop from the network
e_pin = np.hypot(u_pin - u_ref[water], v_pin - v_ref[water]).max()   # the worst velocity error, in units of U
# finite volumes against the reference, at their own faces outside the tube
XU, YU = np.meshgrid(fv["xu"], fv["yu"])
uf = pb.reference_at(ref, XU, YU)[0]; k = ~np.isnan(uf) & ~fv["in_tube_u"]
e_fv = np.abs(fv["u"] - uf)[k].max()

pct = lambda a, b: f"{100 * (a / b - 1):+.2f} %"
print(f"{'':34s}{'pressure drop':>22s}{'drag on the tube':>24s}{'worst speed error':>19s}{'one solution':>14s}{'training':>10s}")
print(f"  {'reference, fitted mesh H/40':32s}{pb.P_SCALE * ref['dp']:11.2f} Pa{'':8s}{pb.F_SCALE * ref['drag']:12.2f} N/m{'':8s}{'-':>15s}{'-':>14s}{'-':>10s}")
print(f"  {'finite volumes, 320 x 80':32s}{pb.P_SCALE * fv['dp']:11.2f} Pa {pct(fv['dp'], ref['dp']):>7s}{pb.F_SCALE * fv['drag']:12.2f} N/m {pct(fv['drag'], ref['drag']):>7s}{e_fv:13.2f} U{t_fv:12.2f} s{'-':>10s}")
print(f"  {'PINN, 4 x 32, stream function':32s}{pb.P_SCALE * dp_pin:11.2f} Pa {pct(dp_pin, ref['dp']):>7s}{pb.F_SCALE * drag_pin:12.2f} N/m {pct(drag_pin, ref['drag']):>7s}{e_pin:13.2f} U{t_pin:12.2f} s{train_time:8.0f} s")

speed = np.full(GX.shape, np.nan); speed[water] = np.hypot(u_pin, v_pin)
fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.4), gridspec_kw=dict(width_ratios=[2.2, 1]))
im = axes[0].contourf(GX, GY, pb.U_MEAN * speed, 24, cmap="viridis")
axes[0].set_aspect("equal"); axes[0].set_xlabel("x / H"); axes[0].set_ylabel("y / H"); axes[0].set_title("the network's mean speed  [m/s]")
fig.colorbar(im, ax=axes[0], shrink=0.8)
j = np.argmin(np.abs(fv["xu"] - 2.0)); yy = np.linspace(0.005, 0.995, 199)   # a section through the wake, x = 2 H
qq = to_tensor(np.c_[np.full(len(yy), 2.0), yy], requires_grad=True)         # the same section for the network
axes[1].plot(pb.U_MEAN * pb.reference_at(ref, np.full(len(yy), 2.0), yy)[0], yy, "k", lw=2.5, label="reference")
axes[1].plot(pb.U_MEAN * fv["u"][:, j], fv["yu"], "--", label="finite volumes")
axes[1].plot(pb.U_MEAN * to_numpy(flow.fields(qq)[0]).ravel(), yy, ":", lw=2.5, label="PINN")
axes[1].set_xlabel("u  [m/s], two tube diameters behind the tube"); axes[1].set_ylabel("y / H"); axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(cc.output_path("Ex09.2_report.png"), dpi=150); plt.show()

**What you should see.** The water speeding up to about 0.60 m/s in the two
gaps beside the tube, and a slow wake behind it that has filled in again by
the outlet. Finite volumes are within 2.86 % on the pressure drop and
4.03 % on the drag, inside the agreed 5 %; the network is within about
0.48 % and 1.00 %. The network's worst velocity error is about 0.02 of the
mean speed; the grid's is 0.26, on the faces next to the staircase. Finite
volumes take about a quarter of a minute and no training; the network needs about two and a half minutes of
training on a CPU, and then gives the field in about a hundredth of a second.

---

## 6 · The inverse problem: the eddy viscosity from twelve probes

Everything above rests on one number that was **chosen**: $\nu_{\mathrm{eff}}$.
Now suppose it is not known, but twelve probes in the wake measured the mean
speed $u$, with a noise of 1 % of the mean speed. Can the same network find
the eddy viscosity?

It is L9.2's *Data Assimilation* slide. One number is trained beside the
weights: $\nu_{\mathrm{eff}} = e^{\beta}$, an exponential so that it cannot
go negative (L9.2's *Prescribe or Learn*). The loss gains one term, the misfit
to the readings, with the weight $w_D$ = 100; the residuals are the ones you
wrote, now with the trainable $\nu_{\mathrm{eff}}$. $\beta$ starts at
$\nu_{\mathrm{eff}}$ = 0.04, twice the true value. The readings are
**synthetic**: the reference, sampled at the probes, with noise added.

**The residuals are divided by $\nu_{\mathrm{eff}}$** (and multiplied by its
starting value, to keep their size). Without that the optimiser has a cheap
way out: it lets the viscosity run to zero, where a flow with a thin layer on
the tube fits twelve probes just as well. Dividing makes that route expensive.
It is the precaution of Ex_08.2, where the residual was divided by $C$.

In [ ]:
u_meas = pb.probe_readings(ref)                               # what the twelve probes read, in units of U, with noise
probes = to_tensor(pb.PROBES, requires_grad=True)             # where they sit; differentiable, the velocity is a derivative
u_m = to_tensor(u_meas.reshape(-1, 1))                        # the readings as a column tensor

NU_START = 0.04                                               # the first guess of nu_eff / (U H): twice the true value
class Inverse(torch.nn.Module):                               # the flow's network and one more trainable number
    def __init__(self):
        super().__init__()
        self.flow = Flow()                                    # a fresh network, the same trial stream function
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(NU_START))))   # nu_eff = exp(beta), starts twice too high
inv = Inverse().to(DEVICE)                                    # the network and beta, on the device
W_D = 100.0                                                   # the weight of the readings
def inverse_loss():
    nu = torch.exp(inv.beta)                                  # the viscosity being identified, always positive
    res_u, res_v = residuals(inv.flow, pts, nu)               # the same residuals, with the trainable viscosity
    s = NU_START / nu                                         # divided by nu, so that nu cannot run to zero
    data = (inv.flow.fields(probes)[0] - u_m).pow(2).mean()   # the misfit at the probes
    return (s * res_u).pow(2).mean() + (s * res_v).pow(2).mean() + boundary_loss(inv.flow) + W_D * data
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(inv, inverse_loss, adam_steps=300, lbfgs_steps=60, lr=2e-3, report_every=0)
inv_time = time.perf_counter() - t0
nu_found = float(torch.exp(inv.beta))
drag_inv, dp_inv = pb.network_numbers(inv.flow.fields, nu_found)   # the two numbers with the viscosity it found
print(f"effective viscosity: found {nu_found * pb.U_MEAN * pb.H_DUCT:.2e} m^2/s, true {pb.NU_EFF_SI:.2e} m^2/s, "
      f"off by {100 * (nu_found / pb.NU_EFF - 1):+.2f} %   ({inv_time:.0f} s)")
print(f"with it: pressure drop {pb.P_SCALE * dp_inv:.2f} Pa ({pct(dp_inv, ref['dp'])}), "
      f"drag {pb.F_SCALE * drag_inv:.2f} N/m ({pct(drag_inv, ref['drag'])})")

**What you should see.** The effective viscosity found within about 3 %
of the true 5.0e-04 m²/s, in about two minutes on a CPU - from twelve noisy
readings and a start that was twice too high - and with it a pressure drop
and a drag within about 2 % and 1 % of the reference.

---

## 7 · What the notebook says

* **The answer is a pressure drop of about 218 Pa and a drag of about 11.4 N
  per metre of tube - for this closure.** Both numbers come from a model of
  the turbulence that is one chosen number. A result like this is reported
  together with the closure that produced it.
* **All three methods agree on the modelled flow.** That says the equations
  were solved correctly. It says nothing about whether the closure is right:
  three solvers can agree on a wrong model.
* **The staircase costs the grid its accuracy.** Finite volumes need
  320 × 80 cells to come within 5 %, because the tube they see is the wrong
  size. The fitted mesh and the network both see a round tube.
* **For the forward problem a classical solver is still the tool.** The
  fitted finite elements give the answer to four figures in five seconds. The
  network needs minutes of training for two figures.
* **The inverse problem is where the network earns its training.** The same
  code, with one more trainable number and one more loss term, found the eddy
  viscosity from twelve probes. A flow code cannot take measurements as an
  input: it would have to be run again and again inside an optimiser.
* **One number was identified because one number was hidden.** A real eddy
  viscosity is a field, and twelve probes do not determine a field: far from
  any probe, many fields fit equally well. That is mini project MP9.2B.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex09.2_report.md` with the results of section 5 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L9.2's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "Why is the mean flow solved for and not the flow itself, and what did the Reynolds number of 25 000 say before anything was computed? (-> L9.2 Q1)": """
""",
    "Which term does the eddy viscosity stand in for, where does it enter your residuals, and what was assumed when it was set to one number for the whole duct? (-> L9.2 Q2)": """
""",
    "Your pressure drop and drag agree with the reference to within a few per cent. What does that agreement show, what does it not show, and what must be reported with the two numbers? (-> L9.2 Q3)": """
""",
    "In the inverse problem, what was trained besides the weights, why as an exponential, how close did twelve probes bring it, and why would twelve probes not give an eddy viscosity that varies in space? (-> L9.2 Q4)": """
""",
    "To conclude: for which task in this notebook would you use the finite-element solver, and for which the network? Say what each would need from you before you trusted its drag. (-> L9.2, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| method | pressure drop | drag on the tube | worst speed error | one solution | training |", "|---|---|---|---|---|---|",
            f"| reference, fitted mesh H/40 | {pb.P_SCALE * ref['dp']:.2f} Pa | {pb.F_SCALE * ref['drag']:.2f} N/m | - | - | - |",
            f"| finite volumes, 320 x 80 | {pb.P_SCALE * fv['dp']:.2f} Pa ({pct(fv['dp'], ref['dp'])}) | {pb.F_SCALE * fv['drag']:.2f} N/m ({pct(fv['drag'], ref['drag'])}) | {e_fv:.2f} U | {t_fv:.2f} s | - |",
            f"| PINN, 4 x 32, stream function | {pb.P_SCALE * dp_pin:.2f} Pa ({pct(dp_pin, ref['dp'])}) | {pb.F_SCALE * drag_pin:.2f} N/m ({pct(drag_pin, ref['drag'])}) | {e_pin:.2f} U | {t_pin:.2f} s | {train_time:.0f} s |",
            "", f"Inverse problem: effective viscosity found {nu_found * pb.U_MEAN * pb.H_DUCT:.2e} m^2/s, true {pb.NU_EFF_SI:.2e} m^2/s, "
                f"off by {100 * (nu_found / pb.NU_EFF - 1):+.2f} %, in {inv_time:.0f} s"]
    out = ["# Ex_09.2 — The Mean Flow Past a Tube", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex09.2_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex09.2_report.md into Ex09.2_report.pdf, with any figure
# saved as Ex09.2_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex09.2_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex09.2_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex09.2_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex09.2_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP9.2A · The wake starts to shed

**The problem.** The flow of this set was steady. Behind a cylinder at a Reynolds number of 100
it is not: vortices peel off one side and then the other, and the wake
oscillates at a frequency set by the flow. Here the problem is the unsteady,
laminar Navier-Stokes equations in $(x, y, t)$, on the confined-cylinder
benchmark every flow code is tested against.

| | |
|---|---|
| channel | 2.2 × 0.41 m, parabolic inflow of mean 1 m/s, no-slip walls, free outlet |
| cylinder | diameter 0.1 m at (0.2, 0.2) m - slightly off the centreline, so shedding starts on its own |
| fluid | $\nu$ = 1e-03 m²/s, Re = 100 |
| reference | Schäfer & Turek (1996), case 2D-2: Strouhal number 0.30, largest drag coefficient 3.23, largest lift coefficient 1.00 |

**Why a plain PINN is not enough.** The answer is periodic, and a network trained on a short window can settle on a
steady wake that satisfies the equations almost as well. Nothing in the loss
asks it to shed.

**What you build.** A PINN for the stream function and pressure in $(x, y, t)$, with
incompressibility built in as in this set, trained over several shedding
periods. You decide how to start it and how to split time, and you measure the
shedding frequency and the forces from your network.

**The ground truth you get.** The benchmark's published values, from converged high-resolution solutions,
are the reference you are scored against. `MP9.2A_truth.npz`, from
`tools/miniprojects/ex092_truth.py`, adds the flow fields of the course's own
solver - a projection method on a staggered grid, the cylinder by direct
forcing - with its error stated below.

**Worked example.** 

| cells | cells per diameter | Strouhal | largest drag coefficient | lift amplitude | one run |
|---|---|---|---|---|---|
| 220 × 41 | 10 | 0.234 | 3.74 | 0.14 | 4 s |
| 440 × 82 | 20 | 0.255 | 3.53 | 0.53 | 44 s |
| 880 × 164 | 40 | 0.272 | 3.37 | 0.80 | 994 s |
| published | | 0.30 | 3.23 | 1.00 | |

The course solver moves towards the benchmark but is first order at the
cylinder's edge, so at 40 cells per diameter it is still 9 % low on the
Strouhal number: its fields show the shape of the wake, and the published
numbers are the ones to meet.

**Requirements.**

* The Strouhal number within **3 %** of 0.30.
* The largest drag coefficient within **5 %** of 3.23, and the largest lift coefficient within **10 %** of 1.00.
* Shedding that starts on its own, over at least five periods.

**What you hand in.** The shedding frequency and the forces against the benchmark's, the wake over one
period, and what your network does if you train it on too short a window.

### MP9.2B · The eddy viscosity from measurements

**The problem.** This set prescribed the eddy viscosity $\nu_t$ - the closure every turbulent
calculation has to guess - as one number, and section 6 found that number
from twelve probes. A real eddy viscosity is a field. Here it is unknown: a
few velocity measurements in the channel, as a wind tunnel would give, are all
there is; find the $\nu_t$ field that makes the mean-flow equations agree
with them.

| | |
|---|---|
| channel | this set's: 4 × 1, a circle of radius 0.2 at (1.2, 0.5), parabolic inflow of mean 1 |
| fluid | $\nu$ = 0.005, and a hidden $\nu_t(x, y)$ up to 0.02 in the wake - a synthetic field chosen for the project |
| probes | $u$ at 12 points, $x$ = 1.6 to 2.8 and $y$ = 0.3 to 0.7, noise 0.01 |

**Why a plain PINN is not enough.** An eddy viscosity that fits the measurements is not necessarily the right one:
far from any probe, many fields fit equally well, and the answer there is the
network's guess.

**What you build.** An inverse PINN: the flow network of this set and a second network for
$\nu_t(x, y)$, kept positive, fitted to the steady mean-flow equations and the
probes. You show where $\nu_t$ is and is not determined.

**The ground truth you get.** `MP9.2B_readings.npz` to fit and `MP9.2B_truth.npz` with the hidden field,
from the same script: the steady flow by the same solver on 320 × 80 cells,
marched until the velocity changes by less than 1e-05 per unit time.

**Worked example.** 

| cells | drag with $\nu_t$ | without | $\nu_t$ moves the probes' $u$ by |
|---|---|---|---|
| 80 × 20 | 1.139 | 1.065 | 0.054 - 0.468 |
| 160 × 40 | 1.073 | 1.004 | 0.043 - 0.637 |
| 320 × 80 | 1.022 | 0.945 | 0.025 - 0.697 |

The eddy viscosity adds about 8 % to the drag on every grid, and moves every
probe by at least twice the noise. The readings and the truth come from the
same 320 × 80 solution, so the problem is consistent even though the drag
itself is still settling by 5 % between grids.

**Requirements.**

* The peak of $\nu_t$ within **25 %**, and where it peaks within **0.2** channel units.
* The probes' velocity reproduced within **0.02**.
* A map of where your $\nu_t$ should not be believed.

**What you hand in.** The recovered $\nu_t$ against the true one, the drag your flow predicts with and
without it, and the map of where it is undetermined.